Import Library

In [1]:
import os
import pandas as pd
import numpy as np

Path Dataset

In [2]:
BASE_DIR = "../data/raw"

NIH_PATH = os.path.join(BASE_DIR, "chestxray14")
CHEXPERT_PATH = os.path.join(BASE_DIR, "chexpert")
TBX11K_PATH = os.path.join(BASE_DIR, "tbx11k")

Harmonisasi NIH

In [3]:
nih_df = pd.read_csv(
    os.path.join(NIH_PATH, "Data_Entry_2017.csv")
)

nih_harmonized = pd.DataFrame()

nih_harmonized["image_path"] = (
    NIH_PATH + "/images/" + nih_df["Image Index"]
)

nih_harmonized["cardiomegaly"] = (
    nih_df["Finding Labels"]
    .str.contains("Cardiomegaly", na=False)
    .astype(int)
)

nih_harmonized["tb"] = np.nan

nih_harmonized["source"] = "nih"

nih_harmonized.head()

,image_path,cardiomegaly,tb,source
0,../data/raw\chestxray14/images/00000001_000.png,1,NaN,nih
1,../data/raw\chestxray14/images/00000001_001.png,1,NaN,nih
2,../data/raw\chestxray14/images/00000001_002.png,1,NaN,nih
3,../data/raw\chestxray14/images/00000002_000.png,0,NaN,nih
4,../data/raw\chestxray14/images/00000003_000.png,0,NaN,nih


Harmonisasi CheXpert

In [4]:
chexpert_df = pd.read_csv(
    os.path.join(CHEXPERT_PATH, "train.csv")
)

chexpert_harmonized = pd.DataFrame()

chexpert_harmonized["image_path"] = (
    "../data/raw/" +
    chexpert_df["Path"].str.replace(
        "CheXpert-v1.0-small",
        "chexpert"
    )
)

chexpert_harmonized["cardiomegaly"] = (
    chexpert_df["Cardiomegaly"]
)

chexpert_harmonized["tb"] = np.nan

chexpert_harmonized["source"] = "chexpert"

chexpert_harmonized.head()

,image_path,cardiomegaly,tb,source
0,../data/raw/chexpert/train/patient00001/study1...,NaN,NaN,chexpert
1,../data/raw/chexpert/train/patient00002/study2...,-1.0,NaN,chexpert
2,../data/raw/chexpert/train/patient00002/study1...,NaN,NaN,chexpert
3,../data/raw/chexpert/train/patient00002/study1...,NaN,NaN,chexpert
4,../data/raw/chexpert/train/patient00003/study1...,NaN,NaN,chexpert


Harmonisasi TBX11K

In [5]:
tb_folder = os.path.join(TBX11K_PATH, "imgs", "tb")
health_folder = os.path.join(TBX11K_PATH, "imgs", "health")
sick_folder = os.path.join(TBX11K_PATH, "imgs", "sick")

tb_images = os.listdir(tb_folder)
health_images = os.listdir(health_folder)
sick_images = os.listdir(sick_folder)

Buat Metadata TB

In [6]:
tb_df = pd.DataFrame({
    "image_path": [
        os.path.join(tb_folder, img)
        for img in tb_images
    ],
    "cardiomegaly": np.nan,
    "tb": 1,
    "source": "tbx11k"
})

Healthy

In [7]:
health_df = pd.DataFrame({
    "image_path": [
        os.path.join(health_folder, img)
        for img in health_images
    ],
    "cardiomegaly": np.nan,
    "tb": 0,
    "source": "tbx11k"
})

Sick

In [8]:
sick_df = pd.DataFrame({
    "image_path": [
        os.path.join(sick_folder, img)
        for img in sick_images
    ],
    "cardiomegaly": np.nan,
    "tb": 0,
    "source": "tbx11k"
})

Gabungkan TBX11K

In [9]:
tbx11k_harmonized = pd.concat([
    tb_df,
    health_df,
    sick_df
])

tbx11k_harmonized.head()

,image_path,cardiomegaly,tb,source
0,../data/raw\tbx11k\imgs\tb\tb0003.png,NaN,1,tbx11k
1,../data/raw\tbx11k\imgs\tb\tb0004.png,NaN,1,tbx11k
2,../data/raw\tbx11k\imgs\tb\tb0005.png,NaN,1,tbx11k
3,../data/raw\tbx11k\imgs\tb\tb0006.png,NaN,1,tbx11k
4,../data/raw\tbx11k\imgs\tb\tb0007.png,NaN,1,tbx11k


Gabungkan Semua Dataset

In [10]:
master_df = pd.concat([
    nih_harmonized,
    chexpert_harmonized,
    tbx11k_harmonized
])

master_df.reset_index(drop=True, inplace=True)

master_df.head()

,image_path,cardiomegaly,tb,source
0,../data/raw\chestxray14/images/00000001_000.png,1.0,NaN,nih
1,../data/raw\chestxray14/images/00000001_001.png,1.0,NaN,nih
2,../data/raw\chestxray14/images/00000001_002.png,1.0,NaN,nih
3,../data/raw\chestxray14/images/00000002_000.png,0.0,NaN,nih
4,../data/raw\chestxray14/images/00000003_000.png,0.0,NaN,nih


Simpan Metadata

In [11]:
save_path = "../data/metadata/harmonized_labels.csv"

os.makedirs("../data/metadata", exist_ok=True)

master_df.to_csv(save_path, index=False)

print("Metadata saved!")

Metadata saved!
